In [1]:
library(tidyverse)
library(tmap)
library(sf)
library(usmap) 
library(janitor) # Highly recommended for cleaning column names

# Load Data 
eyes_hurt_data <- read_csv('Personal Project/search_data/geoMap-3daysafter.csv', skip = 2) %>%
  left_join(
    read_csv('Personal Project/search_data/geoMap-3daysbefore.csv', skip = 2) %>%
      rename(search_volume = 2) %>%
      select(Region, search_volume) %>%
      rename(sv_before = search_volume),
    by = "Region"
  ) %>%
  rename(state = Region) %>%
  rename(sv_after = 2) %>%
  mutate(across(starts_with("sv_"), ~replace_na(., 0))) %>%
  mutate(sv_diff = sv_after - sv_before)%>%
  mutate(sv_per = (sv_after - sv_before)/100) %>%
  mutate(sv_log_fold_change = log((sv_after + 1) / (sv_before + 1)))

map_data <- usmap::us_map(regions = "state") %>% 
  rename(state = full) %>%
  st_transform(4326) %>%
  left_join(eyes_hurt_data, by = "state")

eclipse_central_line <- read_csv('Personal Project/eclipse_path/eclipse_path.csv') %>% 
  filter(!is.na(C_Lon) & !is.na(C_Lat)) %>%
  st_as_sf(coords = c("C_Lon", "C_Lat"), crs = 4326) %>% 
  summarise(do_union = FALSE) %>% 
  st_cast("LINESTRING") %>% 
  st_transform(st_crs(map_data))

── Attaching core tidyverse packages ───────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.0     
── Conflicts ─────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the ]8;;http://conflicted.r-lib.org/conflicted package]8;; to force all conflicts to become errors


proj_create: Cannot find proj.db
proj_create: no database context specified


There appears to be a problem with the PROJ installation
Linking to GEOS 3.13.0, GDAL 3.8.5, PROJ 9.5.1; sf_use_s2() is TRUE

Attaching package: ‘janitor’

The following objects are masked from ‘package:stats’:

    chisq.test, fisher.test

Rows: 51 Columns: 2
── Column specification ─────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (1): Region
dbl (1): eyes hurt: (4/7/24 - 4/9/24)

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 51 Columns: 2
── Column specification ─────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (1): Region
dbl (1): eyes hurt: (4/4/24 - 4/6/24)

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 99 Co

In [2]:
state_proj <- st_transform(map_data, 5070) 
eclipse_proj <- st_transform(eclipse_central_line, 5070)

state_proj$distance_km <- as.numeric(
  st_distance(state_proj, eclipse_proj)
) / 1000


# Correlation Tests